# Estructuras lineales — implementación de referencia

Esta es **la implementación que vamos a usar en el resto del curso**. Ustedes están construyendo
la suya en el taller de la unidad, y está bien que sea distinta: esta es la mía, y es la que
voy a dar por supuesta de aquí en adelante.

No es una clase para estudiar línea por línea, es una **herramienta para usar**. Cópiala al
principio de tus cuadernos y sigue trabajando.

## Qué trae

| Clase | Para qué | Cómo se crea |
|---|---|---|
| `Pila` | LIFO | `Pila()` · `Pila([1,2,3])` · `Pila(capacidad=5)` |
| `Cola` | FIFO, lineal **o** circular | `Cola()` · `Cola([1,2,3])` · `Cola(capacidad=5)` |
| `ColaCircular` | lo mismo que `Cola` con capacidad, pero el nombre lo dice | `ColaCircular(5)` |

## Una decisión de diseño que vale la pena explicar

En el taller les pedí **tres** clases separadas. Acá hay **dos y media**, y a propósito.

Una cola circular no es una estructura distinta de una cola: es **una cola con cupo fijo**. Todo
lo demás —FIFO, `encolar`, `desencolar`, `frente`— es idéntico. Entonces en vez de escribir dos
veces el mismo código, `Cola` recibe una `capacidad` opcional:

* `Cola()` → **lineal**: crece sin límite.
* `Cola(capacidad=4)` → **circular**: cupo fijo, y `encolar` lanza `OverflowError` cuando se llena.

`ColaCircular` existe igual, pero son tres líneas: una subclase que vuelve obligatoria la
capacidad. Así el nombre dice lo que es cuando lo necesitas.

## El aislamiento

Las dos clases guardan sus datos en `__datos`, con doble guion bajo. Desde fuera **no hay forma
cómoda de llegar a la colección**: no hay atributos públicos, ninguna propiedad entrega la
estructura viva, y `__iter__` recorre una **copia**. Si intentas `mi_pila.datos.append(9)`, falla.

---

## Lo único que hay que importar


In [ ]:
from collections import deque


---

## `Pila`

In [ ]:
class Pila:
    """Pila LIFO: el ultimo en entrar es el primero en salir.

    Pila()            -> pila que crece sin limite
    Pila(capacidad=3) -> pila con cupo fijo: apilar lanza OverflowError si esta llena
    """

    def __init__(self, iterable=None, capacidad=None):
        if capacidad is not None and capacidad < 1:
            raise ValueError("la capacidad debe ser al menos 1")
        self.__capacidad = capacidad
        self.__datos = deque()
        for elemento in (iterable or ()):
            self.apilar(elemento)

    # --- estado ---
    @property
    def capacidad(self):
        """Cupo maximo, o None si la pila crece sin limite."""
        return self.__capacidad

    def esta_vacia(self):
        return len(self.__datos) == 0

    def esta_llena(self):
        return self.__capacidad is not None and len(self.__datos) == self.__capacidad

    # --- operaciones ---
    def apilar(self, elemento):
        """Pone un elemento en la cima."""
        if self.esta_llena():
            raise OverflowError(f"la pila esta llena (capacidad {self.__capacidad})")
        self.__datos.append(elemento)

    def desapilar(self):
        """Saca y retorna el elemento de la cima."""
        if self.esta_vacia():
            raise IndexError("desapilar de una pila vacia")
        return self.__datos.pop()

    @property
    def cima(self):
        """El elemento de la cima, sin sacarlo."""
        if self.esta_vacia():
            raise IndexError("cima de una pila vacia")
        return self.__datos[-1]

    def vaciar(self):
        """Deja la pila sin elementos."""
        self.__datos.clear()

    # --- protocolos ---
    def __len__(self):
        return len(self.__datos)

    def __bool__(self):
        return len(self.__datos) > 0

    def __contains__(self, elemento):
        return elemento in self.__datos

    def __iter__(self):
        """Recorre de la cima hacia la base, sobre una copia."""
        return iter(tuple(reversed(self.__datos)))

    def __repr__(self):
        tope = "" if self.__capacidad is None else f", capacidad={self.__capacidad}"
        return f"Pila({list(self.__datos)!r}{tope})"

## `Cola`

In [ ]:
class Cola:
    """Cola FIFO: el primero en entrar es el primero en salir.

    Cola()            -> cola lineal: crece sin limite
    Cola(capacidad=3) -> cola CIRCULAR: cupo fijo, reutiliza el espacio que se libera
                         al frente. encolar lanza OverflowError si esta llena.
    """

    def __init__(self, iterable=None, capacidad=None):
        if capacidad is not None and capacidad < 1:
            raise ValueError("la capacidad debe ser al menos 1")
        self.__capacidad = capacidad
        self.__datos = deque()
        for elemento in (iterable or ()):
            self.encolar(elemento)

    # --- estado ---
    @property
    def capacidad(self):
        """Cupo maximo, o None si la cola crece sin limite."""
        return self.__capacidad

    @property
    def es_circular(self):
        """True si tiene cupo fijo."""
        return self.__capacidad is not None

    def esta_vacia(self):
        return len(self.__datos) == 0

    def esta_llena(self):
        return self.__capacidad is not None and len(self.__datos) == self.__capacidad

    # --- operaciones ---
    def encolar(self, elemento):
        """Pone un elemento al final."""
        if self.esta_llena():
            raise OverflowError(f"la cola esta llena (capacidad {self.__capacidad})")
        self.__datos.append(elemento)

    def encolar_pisando(self, elemento):
        """Encola y, si esta llena, descarta el mas viejo. Retorna lo descartado o None."""
        descartado = self.desencolar() if self.esta_llena() else None
        self.__datos.append(elemento)
        return descartado

    def desencolar(self):
        """Saca y retorna el elemento del frente."""
        if self.esta_vacia():
            raise IndexError("desencolar de una cola vacia")
        return self.__datos.popleft()

    @property
    def frente(self):
        """El elemento del frente, sin sacarlo."""
        if self.esta_vacia():
            raise IndexError("frente de una cola vacia")
        return self.__datos[0]

    @property
    def final(self):
        """El ultimo elemento, sin sacarlo."""
        if self.esta_vacia():
            raise IndexError("final de una cola vacia")
        return self.__datos[-1]

    def rotar(self):
        """Manda el del frente al final y lo retorna. Util para Round-Robin."""
        elemento = self.desencolar()
        self.__datos.append(elemento)
        return elemento

    def vaciar(self):
        """Deja la cola sin elementos."""
        self.__datos.clear()

    # --- protocolos ---
    def __len__(self):
        return len(self.__datos)

    def __bool__(self):
        return len(self.__datos) > 0

    def __contains__(self, elemento):
        return elemento in self.__datos

    def __iter__(self):
        """Recorre del frente hacia el final, sobre una copia."""
        return iter(tuple(self.__datos))

    def __repr__(self):
        tipo = "circular" if self.es_circular else "lineal"
        tope = "" if self.__capacidad is None else f", capacidad={self.__capacidad}"
        return f"Cola({list(self.__datos)!r}{tope})  # {tipo}"

## `ColaCircular`

In [ ]:
class ColaCircular(Cola):
    """Una Cola con capacidad obligatoria. Existe para que el nombre diga lo que es."""

    def __init__(self, capacidad, iterable=None):
        super().__init__(iterable, capacidad=capacidad)

---

# Cómo se usa

## La pila

In [ ]:
p = Pila()
p.apilar("A")
p.apilar("B")
p.apilar("C")
print(p)

In [ ]:
print("cima      :", p.cima)        # sin sacarlo, y sin parentesis
print("desapilar :", p.desapilar())
print("quedan    :", len(p), "->", p)

In [ ]:
# Recorre de la CIMA hacia la base
for elemento in p:
    print(" -", elemento)

In [ ]:
# Una pila con tope
limitada = Pila(capacidad=2)
limitada.apilar(1)
limitada.apilar(2)

try:
    limitada.apilar(3)
except OverflowError as e:
    print("Error:", e)

## La cola lineal

In [ ]:
c = Cola(["Ana", "Luis", "Sara"])
print(c)
print("frente:", c.frente, "| final:", c.final)

In [ ]:
print("se atiende:", c.desencolar())
print("quedan    :", list(c))

## La cola circular

In [ ]:
cc = Cola(capacidad=4)
for letra in "ABCD":
    cc.encolar(letra)

print(cc)
print("llena:", cc.esta_llena())

In [ ]:
try:
    cc.encolar("E")
except OverflowError as e:
    print("Error:", e)

In [ ]:
# Liberamos dos espacios y volvemos a encolar: los nuevos dan la vuelta
cc.desencolar()
cc.desencolar()
cc.encolar("E")
cc.encolar("F")

print(list(cc))     # ['C', 'D', 'E', 'F']

## Dos atajos que vas a usar

`encolar_pisando` descarta el más viejo cuando está llena, en vez de rechazar. Es lo que
necesitas para *"las últimas N lecturas"*.

In [ ]:
ultimas = Cola(capacidad=3)

for lectura in [22.1, 22.5, 22.3, 22.8, 23.0]:
    descartada = ultimas.encolar_pisando(lectura)
    print(f"  entra {lectura} -> {list(ultimas)}   descartada: {descartada}")

`rotar` manda el del frente al final. Es Round-Robin en una línea.

In [ ]:
turnos = Cola(["P1", "P2", "P3"])

for _ in range(5):
    print("  atiende a", turnos.rotar())

---

# Tabla de referencia

## `Pila`

| | Qué hace | Si está vacía / llena |
|---|---|---|
| `apilar(x)` | pone `x` en la cima | `OverflowError` si tiene capacidad y está llena |
| `desapilar()` | saca y retorna el de la cima | `IndexError` |
| `cima` | el de la cima, sin sacarlo | `IndexError` |
| `esta_vacia()` / `esta_llena()` | `True` / `False` | — |
| `capacidad` | el cupo, o `None` si crece | — |
| `vaciar()` | la deja sin elementos | — |
| `len(p)` · `bool(p)` · `x in p` · `for x in p` · `print(p)` | protocolos | `for` va de la cima a la base |

## `Cola` y `ColaCircular`

| | Qué hace | Si está vacía / llena |
|---|---|---|
| `encolar(x)` | pone `x` al final | `OverflowError` si es circular y está llena |
| `encolar_pisando(x)` | encola descartando el más viejo; retorna lo descartado | nunca falla |
| `desencolar()` | saca y retorna el del frente | `IndexError` |
| `frente` / `final` | los extremos, sin sacarlos | `IndexError` |
| `rotar()` | manda el del frente al final y lo retorna | `IndexError` |
| `esta_vacia()` / `esta_llena()` | `True` / `False` | — |
| `capacidad` / `es_circular` | el cupo y si lo tiene | — |
| `vaciar()` | la deja sin elementos | — |
| `len(c)` · `bool(c)` · `x in c` · `for x in c` · `print(c)` | protocolos | `for` va del frente al final |

---

# Cómo reutilizarla

Lo más simple: **copia las cuatro celdas de código** (el `import` y las tres clases) al
principio de tu cuaderno.

Si prefieres no repetirlas, guárdalas en un archivo `estructuras.py` junto a tu notebook y escribe:

```python
from estructuras import Pila, Cola, ColaCircular
```

```{tip}
De aquí en adelante, cuando un enunciado diga *"usa una pila"*, se refiere a esta. En la Unidad 2
la vas a necesitar para *backtracking*, y en la Unidad 3 la cola aparece en los recorridos.
```